In [62]:
from midiutil import MIDIFile
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

def vector_xyz(state):
    """
    Function to translate a complex state into a 3D real vector.
    """
    
# Extract the complex amplitudes (alpha and beta)
    alpha = state.data[0]
    beta = state.data[1]

    # Compute Cartesian (x, y, z) coordinates 
    x = 2 * np.real(np.conj(alpha) * beta)
    y = 2 * np.imag(np.conj(alpha) * beta)
    z = np.abs(alpha)**2 - np.abs(beta)**2

    return([x, y, z])

def vector_spherical(state):
    """
    Function to translate a Cartesian state into spherical coordinates.
    """
    x, y, z = state

    r = np.sqrt(x**2 + y**2 + z**2)
    if r == 0:
        return [0.0, 0.0, 0.0]

    theta = np.arccos(z / r)
    phi = np.arctan2(y, x)

    return [r, theta, phi]


def gates(gate_list, qc = QuantumCircuit(1)):
    """
    Function to apply a list of gates to a qubit.
    """
    states = []

    # Initialise a 0 state qubit.
    qc.initialize('0', 0)
    
    for gate_name in gate_list:
        # Check if the QuantumCircuit object has the requested gate method
        if hasattr(qc, gate_name):
            # Fetch the method (e.g., circuit.h, circuit.cx)
            gate_method = getattr(qc, gate_name)
            # Call the method, unpacking the target qubits
            gate_method(0)
            state = vector_xyz(Statevector(qc))
            states.append(state)
        else:
            print(f"Warning: Gate '{gate_name}' is not a recognized Qiskit method.")

    return states

# Function to find the closest key's value based on a midpoint range.
def get_closest(value, target_dict):
    keys_array = np.array(list(target_dict.keys()))
    values_array = np.array(list(target_dict.values()))
    
    # Find the index of the key with the smallest absolute distance
    closest_idx = np.abs(keys_array - value).argmin()
    return int(values_array[closest_idx])


In [63]:
# Create dictionaries to map state (r, theta, phi) to integer pitch over middle 7 octaves.
phi_dict = {}
theta_dict = {}
# Assumes normalised state with r = 1.

# Theta => octave.
theta_dict = {
    (np.pi / 4): 1,
    (np.pi * 2 / 4): 2,
    (np.pi * 3 / 4): 3,
    (np.pi * 4 / 4): 4,
    (np.pi * 5 / 4): 5,
    (np.pi * 6 / 4): 6,
    (np.pi * 7 / 4): 7
}

for j in range(24, 36): # Phi = > notes.
    phi_dict.update({(((j - 24) / 6) * np.pi) : j})



In [64]:
def theta_and_phi(states):
    notes = []
    """
    Function to combine theta and phi information.
    """
    for i in range(len(states)):

        theta = vector_spherical(states[i])[1]
        phi = vector_spherical(states[i])[2]
        print(theta, phi)
        
        if theta in theta_dict:
            theta_note = theta_dict[theta]
        else:
            theta_note = get_closest(theta, theta_dict)
            
    
        if phi in phi_dict:
            phi_note = phi_dict[phi]
        else:
            phi_note = get_closest(phi, phi_dict)
        print(theta_note, phi_note)

        note = ((theta_note - 1) * 12) + phi_note
        notes.append(note)
        
    return notes
    

In [67]:
gate_list = list(input("Please list gates required."))
states = gates(gate_list)
notes = theta_and_phi(states)
print(notes)
        


Please list gates required. xx


3.141592653589793 -0.0
4 24
0.0 0.0
1 24
[60, 24]


In [66]:
MyMIDI = MIDIFile(1)

track = 0
time = 0

MyMIDI.addTrackName(track, time, "Test track")
MyMIDI.addTempo(track, time, 120)

track    = 0   # Track numbers are zero-origined
channel  = 0   # MIDI channel number -> number of qubits.
duration = 1   # In beats
volume   = 100 # 0-127, 127 being full volume

for i in range(len(notes)):
    pitch = notes[i]
    time = i
    MyMIDI.addNote(track,channel,pitch,time,duration,volume)


with open("mymidifile.midi", 'wb') as output_file:
    MyMIDI.writeFile(output_file)